# Phase 1 — Baseline eval (wmdp_bio + mmlu)

Run in Colab with a GPU runtime (`Runtime` -> `Change runtime type` -> GPU).

This writes out the project's actual `src/common.py`, `src/eval.py`, and `configs/default.yaml` (kept in sync with the local repo), then runs the real CLI: `python -m src.eval` on Qwen2.5-1.5B-Instruct, scoring wmdp_bio (full) and mmlu (500-sample subset for speed).

Safety reminder: wmdp_bio is used only as a scoring benchmark here — we never read or surface its questions as content, only the resulting accuracy score.

In [ ]:
!nvidia-smi
!pip install -q transformers==5.14.1 datasets==5.0.0 accelerate==1.14.0 peft==0.19.1 lm-eval==0.4.12

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile src/common.py
"""Shared helpers: config loading, seeding, model loading, IO."""
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import yaml

REPO_ROOT = Path(__file__).resolve().parent.parent
DEFAULT_CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"


def load_config(config_path=None, overrides=None):
    """Load YAML config, deep-merge `overrides` (e.g. CLI flags) on top."""
    path = Path(config_path) if config_path else DEFAULT_CONFIG_PATH
    if not path.exists():
        raise FileNotFoundError(f"Config file not found: {path}")
    with open(path) as f:
        config = yaml.safe_load(f)
    if overrides:
        _deep_update(config, overrides)
    return config


def _deep_update(base, overrides):
    for key, value in overrides.items():
        if value is None:
            continue
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def resolve_dtype(name):
    return {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}[name]


def library_versions():
    import accelerate
    import datasets
    import transformers

    versions = {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "accelerate": accelerate.__version__,
    }
    try:
        import lm_eval

        versions["lm_eval"] = lm_eval.__version__
    except ImportError:
        pass
    return versions


def load_model_and_tokenizer(model_id, dtype="bfloat16", device=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = device or resolve_device()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=resolve_dtype(dtype),
        device_map=device if device == "cuda" else None,
    )
    if device != "cuda":
        model = model.to(device)
    model.eval()
    return model, tokenizer


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


def append_experiment_index(row, index_path=None):
    """Append a row (timestamp, phase, config hash, key metric) to the experiments index."""
    index_path = Path(index_path) if index_path else REPO_ROOT / "results" / "experiments_index.jsonl"
    index_path.parent.mkdir(parents=True, exist_ok=True)
    with open(index_path, "a") as f:
        f.write(json.dumps(row) + "\n")

In [ ]:
%%writefile src/eval.py
"""Run lm-eval on a model for the given tasks; save a normalized results JSON.

Usage:
    python -m src.eval --model <hf-id-or-path> --tasks wmdp_bio,mmlu --limit <int|None> --out results/<name>.json
"""
import argparse
import datetime
import sys

from src.common import library_versions, load_config, resolve_device, save_json, set_seed


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True, help="HF model id or local path")
    p.add_argument("--tasks", required=True, help="Comma-separated lm-eval task names")
    p.add_argument("--limit", type=int, default=None, help="Sample limit per task (applies to all tasks given)")
    p.add_argument("--mmlu-limit", type=int, default=None, help="Override limit specifically for the mmlu task")
    p.add_argument("--out", required=True, help="Output JSON path")
    p.add_argument("--config", default=None, help="Path to YAML config (defaults to configs/default.yaml)")
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--batch-size", default="auto")
    return p.parse_args()


def extract_metric(task_results, preferred=("acc,none", "acc_norm,none", "acc", "acc_norm")):
    for key in preferred:
        if key in task_results:
            return key, task_results[key]
    # fall back to any key ending in a non-stderr accuracy-like metric
    for key, value in task_results.items():
        if not key.endswith("_stderr,none") and not key.endswith("_stderr") and "stderr" not in key:
            return key, value
    raise KeyError(f"No accuracy-like metric found in task results: {list(task_results.keys())}")


def extract_stderr(task_results, metric_key):
    base = metric_key.split(",")[0]
    suffix = "," + metric_key.split(",", 1)[1] if "," in metric_key else ""
    for candidate in (f"{base}_stderr{suffix}", f"{base}_stderr"):
        if candidate in task_results:
            return task_results[candidate]
    return None


def run_eval(model, tasks, limit=None, per_task_limit=None, seed=0, batch_size="auto"):
    import lm_eval
    from lm_eval.models.huggingface import HFLM

    per_task_limit = per_task_limit or {}
    lm = HFLM(pretrained=model, device=resolve_device(), batch_size=batch_size)

    results_out = {}
    for task in tasks:
        task_limit = per_task_limit.get(task, limit)
        raw = lm_eval.simple_evaluate(
            model=lm,
            tasks=[task],
            limit=task_limit,
            random_seed=seed,
            numpy_random_seed=seed,
            torch_random_seed=seed,
        )
        task_results = raw["results"][task]
        metric_key, acc = extract_metric(task_results)
        stderr = extract_stderr(task_results, metric_key)
        n = task_results.get("sample_count")
        if isinstance(n, dict):
            n = n.get(metric_key, next(iter(n.values()), None))
        if n is None:
            n = raw.get("n-samples", {}).get(task, {}).get("effective")
        results_out[task] = {"acc": acc, "acc_stderr": stderr, "n": n, "metric": metric_key}
    return results_out


def main():
    args = parse_args()
    config = load_config(args.config)
    seed = args.seed if args.seed is not None else config.get("seed", 0)
    set_seed(seed)

    tasks = [t.strip() for t in args.tasks.split(",") if t.strip()]
    per_task_limit = {}
    if args.mmlu_limit is not None and "mmlu" in tasks:
        per_task_limit["mmlu"] = args.mmlu_limit
    elif "mmlu" in tasks and args.limit is None:
        cfg_mmlu_limit = config.get("eval", {}).get("mmlu_limit")
        if cfg_mmlu_limit is not None:
            per_task_limit["mmlu"] = cfg_mmlu_limit

    print(f"Evaluating model={args.model} tasks={tasks} limit={args.limit} "
          f"per_task_limit={per_task_limit} seed={seed}", file=sys.stderr)

    results = run_eval(
        model=args.model,
        tasks=tasks,
        limit=args.limit,
        per_task_limit=per_task_limit,
        seed=seed,
        batch_size=args.batch_size,
    )

    import lm_eval

    output = {
        "model": args.model,
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "lm_eval_version": lm_eval.__version__,
        "tasks": results,
        "config": {
            "tasks": tasks,
            "limit": args.limit,
            "per_task_limit": per_task_limit,
            "seed": seed,
            "batch_size": args.batch_size,
            "library_versions": library_versions(),
        },
    }
    out_path = save_json(args.out, output)
    print(f"Wrote {out_path}")
    for task, r in results.items():
        print(f"  {task}: acc={r['acc']:.4f} (stderr={r['acc_stderr']}) n={r['n']} metric={r['metric']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/default.yaml
model:
  base: "Qwen/Qwen2.5-1.5B-Instruct"
  fallback: "Qwen/Qwen2.5-3B-Instruct"
  dtype: "bfloat16"
  max_seq_len: 512

eval:
  tasks: ["wmdp_bio", "mmlu"]
  mmlu_limit: 500        # subset for speed; null for full
  wmdp_bio_limit: null   # wmdp_bio test set is small; run in full

seed: 0

rmu:
  layer_id: null         # set to ~40-60% depth after inspecting the model (28 layers -> ~11-17)
  steering_coeff: 6.5    # SWEEP {2, 6.5, 20, 60}
  alpha: 1200            # retain weight; tune to preserve MMLU
  lr: 5.0e-5
  max_steps: 200
  batch_size: 4

attack:
  data: "<benign general-domain dataset>"
  learning_rates: [1.0e-5, 5.0e-5]
  steps_grid: [0, 25, 50, 100, 200]
  batch_size: 4
  method: "full"         # "full" | "lora"

In [ ]:
!python -m src.eval --model Qwen/Qwen2.5-1.5B-Instruct --tasks wmdp_bio,mmlu --mmlu-limit 500 --batch-size auto --out results/baseline.json

In [ ]:
# Print the full JSON so it can be copied back to Claude Code.
print(open('results/baseline.json').read())